In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, time, timedelta
from SynthSpread.spreadviewer_class import SpreadSingle, SpreadViewerData, norm_coeff
from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Strategies.MultipleMarketsIntensity_class import MultiTradeIntensity as TI
from Math.ti_class import TI_class, VI_class, TR_class
from Math.lm_class import kalman, LinearModel
from Math.accumfeatures import EMA, MA, MSTD, DifferentialEMA, DerivativeEMA
from Strategies.IntensityHawkes_strategy.model_class import HawkesIntensity
from Strategies.IntensityHawkes_strategy.backtest_class import BacktestIB
from Strategies.IntensityHawkes_strategy.strategy_class import StrategyHI, VolumeClass
tol=(1e-1)/2

In [2]:
import sys,os
sys.path.append(r'C:/data/EnergyTrading/Python/')

import cx_Oracle
try:
    cx_Oracle.init_oracle_client(lib_dir=r"C:\Users\andrej\Downloads\instantclient_21_11")
except:
    pass


# Data Loading

## Using Trayport DA

In [3]:
params_dict = {}

params_dict['tenor_list'] = ['m']
params_dict['tn1_list'] = [1]
params_dict['mkt_list'] = ['de'] * len(params_dict['tenor_list'])
params_dict['tn2_list'] = []
params_dict['prod'] = 'base'
params_dict['venue_list'] = ['eex']*len(params_dict['mkt_list'])
params_dict['start_date'] = datetime(2025, 2, 26)
params_dict['end_date'] = datetime(2025, 7, 16)
params_dict['ns'] = 2

# Fetch trades and best orders for the curve
assembler = TPDataAssembly(source='trayport', user='matej')
# assembler.set_start_end_time(start=[10,0,0], end=[12,0,0])    
trades_dict = assembler.get_data(params_dict, target_data='trades')
#assembler.set_data_source('database')
ba_dict = assembler.get_data(params_dict, target_data='best_orders')

assert ba_dict.keys() == trades_dict.keys(), "Curve doesn't fit for both trades and best_orders"

trades = pd.DataFrame()
ba = pd.DataFrame()
products = []
for key in trades_dict.keys():
    ba_aux = ba_dict[key].copy()
    trade_aux = trades_dict[key].copy()
    trade_aux.columns = [a + '_' + key for a in trade_aux.columns]
    ba_aux.columns = [a + '_' + key for a in ba_aux.columns]
    if trades.empty:
        trades = trade_aux.copy()
    else:
        trades = pd.concat([trades, trade_aux])
    if ba.empty:
        ba = ba_aux.copy()
    else:
        ba = pd.concat([ba, ba_aux])
    products.append(key)
trades.sort_index(inplace=True)
ba.sort_index(inplace=True)
ba.index.name='datetime'

ti_inst = TI(trades, ba, products)

ti_inst.prepare_data()


data_raw = ti_inst.data

data = data_raw[data_raw['broker_id_dem1']==1441][['price_dem1', 'volume_dem1','bidbestprice_dem1',
                  'askbestprice_dem1', 'mid_dem1', 'trade_side_dem1']].copy()

# data = data_raw[['price_dem1', 'volume_dem1','bidbestprice_dem1',
#                    'askbestprice_dem1', 'mid_dem1', 'trade_side_dem1']].copy()
    
data.columns = [a.split('_')[0] for a in data.columns]
data.columns = ['trd_price', 'volume', 'bid_price', 'ask_price', 'mid_price', 'trd_side']

data.to_csv(r's:\Algo\Files\andrej\Data\int_data_lead_production_sample.csv')

https://referencedata.trayport.com/instruments
Duration: 1.405s
https://analytics.trayport.com/api/trades?from=2025-02-26T07%3A00%3A00Z&until=2025-02-26T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=255&ContractType=SinglePeriod
Duration: 1.519s
https://referencedata.trayport.com/instruments
Duration: 1.207s
https://analytics.trayport.com/api/trades?from=2025-02-27T07%3A00%3A00Z&until=2025-03-27T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=256&ContractType=SinglePeriod
Duration: 2.277s
https://referencedata.trayport.com/instruments
Duration: 1.100s
https://analytics.trayport.com/api/trades?from=2025-03-28T07%3A00%3A00Z&until=2025-04-28T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=257&ContractType=SinglePeriod
Duration: 1.908s
https://referencedata.trayport.com/instruments
Duration: 1.188s
https://analytics.trayport.com/api/trades?from=2025-04-29T06%3A00%3A00Z&until=2025-05-28T18%3A00%3A00Z&instrumentId=1064171

https://referencedata.trayport.com/instruments
Duration: 1.304s
https://analytics.trayport.com/api/orders/book?from=2025-03-26T07%3A00%3A00Z&until=2025-03-26T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=256&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 6.222s
https://referencedata.trayport.com/instruments
Duration: 1.257s
https://analytics.trayport.com/api/orders/book?from=2025-03-27T07%3A00%3A00Z&until=2025-03-27T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=256&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 5.186s
https://referencedata.trayport.com/instruments
Duration: 1.120s
https://analytics.trayport.com/api/orders/book?from=2025-03-28T07%3A00%3A00Z&until=2025-03-28T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=257&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.043s
http

https://referencedata.trayport.com/instruments
Duration: 1.230s
https://analytics.trayport.com/api/orders/book?from=2025-04-30T06%3A00%3A00Z&until=2025-04-30T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=258&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 5.099s
https://referencedata.trayport.com/instruments
Duration: 1.133s
https://analytics.trayport.com/api/orders/book?from=2025-05-01T06%3A00%3A00Z&until=2025-05-01T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=258&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 1.556s
https://referencedata.trayport.com/instruments
Duration: 1.339s
https://analytics.trayport.com/api/orders/book?from=2025-05-02T06%3A00%3A00Z&until=2025-05-02T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=258&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.558s
http

https://referencedata.trayport.com/instruments
Duration: 1.244s
https://analytics.trayport.com/api/orders/book?from=2025-06-04T06%3A00%3A00Z&until=2025-06-04T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=259&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.566s
https://referencedata.trayport.com/instruments
Duration: 1.368s
https://analytics.trayport.com/api/orders/book?from=2025-06-05T06%3A00%3A00Z&until=2025-06-05T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=259&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 6.312s
https://referencedata.trayport.com/instruments
Duration: 1.446s
https://analytics.trayport.com/api/orders/book?from=2025-06-06T06%3A00%3A00Z&until=2025-06-06T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=259&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.807s
http

https://referencedata.trayport.com/instruments
Duration: 2.225s
https://analytics.trayport.com/api/orders/book?from=2025-07-09T06%3A00%3A00Z&until=2025-07-09T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=260&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 8.289s
https://referencedata.trayport.com/instruments
Duration: 1.434s
https://analytics.trayport.com/api/orders/book?from=2025-07-10T06%3A00%3A00Z&until=2025-07-10T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=260&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 6.558s
https://referencedata.trayport.com/instruments
Duration: 5.179s
https://analytics.trayport.com/api/orders/book?from=2025-07-11T06%3A00%3A00Z&until=2025-07-11T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=260&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 6.922s
http

In [4]:
params_dict = {}

params_dict['tenor_list'] = ['m']
params_dict['tn1_list'] = [2]
params_dict['mkt_list'] = ['de'] * len(params_dict['tenor_list'])
params_dict['tn2_list'] = []
params_dict['prod'] = 'base'
params_dict['venue_list'] = ['eex']*len(params_dict['mkt_list'])
params_dict['start_date'] = datetime(2025, 2, 26)
params_dict['end_date'] = datetime(2025, 7, 16)
params_dict['ns'] = 2

# Fetch trades and best orders for the curve
assembler = TPDataAssembly(source='trayport', user='matej')
# assembler.set_start_end_time(start=[10,0,0], end=[12,0,0])    
trades_dict = assembler.get_data(params_dict, target_data='trades')
#assembler.set_data_source('database')
ba_dict = assembler.get_data(params_dict, target_data='best_orders')

assert ba_dict.keys() == trades_dict.keys(), "Curve doesn't fit for both trades and best_orders"

trades = pd.DataFrame()
ba = pd.DataFrame()
products = []
for key in trades_dict.keys():
    ba_aux = ba_dict[key].copy()
    trade_aux = trades_dict[key].copy()
    trade_aux.columns = [a + '_' + key for a in trade_aux.columns]
    ba_aux.columns = [a + '_' + key for a in ba_aux.columns]
    if trades.empty:
        trades = trade_aux.copy()
    else:
        trades = pd.concat([trades, trade_aux])
    if ba.empty:
        ba = ba_aux.copy()
    else:
        ba = pd.concat([ba, ba_aux])
    products.append(key)
trades.sort_index(inplace=True)
ba.sort_index(inplace=True)
ba.index.name='datetime'
trades.index.name ='datetime'


ti_inst = TI(trades, ba, products)

ti_inst.prepare_data()


data_raw = ti_inst.data

data = data_raw[~(data_raw['price_dem2'].notnull() & (data_raw['broker_id_dem2'] != 1441))][['price_dem2', 'volume_dem2','bidbestprice_dem2',
                  'askbestprice_dem2', 'mid_dem2', 'trade_side_dem2']].copy()

# data = data_raw[['price_dem2', 'volume_dem2','bidbestprice_dem2',
#                    'askbestprice_dem2', 'mid_dem2', 'trade_side_dem2']].copy()
    
data.columns = [a.split('_')[0] for a in data.columns]
data.columns = ['trd_price', 'volume', 'bid_price', 'ask_price', 'mid_price', 'trd_side']

data.to_csv(r's:\Algo\Files\andrej\Data\int_data_lag_production_sample.csv')

https://referencedata.trayport.com/instruments
Duration: 1.922s
https://analytics.trayport.com/api/trades?from=2025-02-26T07%3A00%3A00Z&until=2025-02-26T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=256&ContractType=SinglePeriod
Duration: 1.416s
https://referencedata.trayport.com/instruments
Duration: 2.341s
https://analytics.trayport.com/api/trades?from=2025-02-27T07%3A00%3A00Z&until=2025-03-27T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=257&ContractType=SinglePeriod
Duration: 1.512s
https://referencedata.trayport.com/instruments
Duration: 1.657s
https://analytics.trayport.com/api/trades?from=2025-03-28T07%3A00%3A00Z&until=2025-04-28T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=258&ContractType=SinglePeriod
Duration: 1.559s
https://referencedata.trayport.com/instruments
Duration: 1.151s
https://analytics.trayport.com/api/trades?from=2025-04-29T06%3A00%3A00Z&until=2025-05-28T18%3A00%3A00Z&instrumentId=1064171

https://referencedata.trayport.com/instruments
Duration: 1.120s
https://analytics.trayport.com/api/orders/book?from=2025-03-26T07%3A00%3A00Z&until=2025-03-26T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=257&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 5.983s
https://referencedata.trayport.com/instruments
Duration: 1.383s
https://analytics.trayport.com/api/orders/book?from=2025-03-27T07%3A00%3A00Z&until=2025-03-27T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=257&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.520s
https://referencedata.trayport.com/instruments
Duration: 18.918s
https://analytics.trayport.com/api/orders/book?from=2025-03-28T07%3A00%3A00Z&until=2025-03-28T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=258&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.973s
htt

https://referencedata.trayport.com/instruments
Duration: 1.751s
https://analytics.trayport.com/api/orders/book?from=2025-04-30T06%3A00%3A00Z&until=2025-04-30T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=259&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 5.031s
https://referencedata.trayport.com/instruments
Duration: 1.801s
https://analytics.trayport.com/api/orders/book?from=2025-05-01T06%3A00%3A00Z&until=2025-05-01T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=259&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 1.785s
https://referencedata.trayport.com/instruments
Duration: 1.176s
https://analytics.trayport.com/api/orders/book?from=2025-05-02T06%3A00%3A00Z&until=2025-05-02T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=259&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.194s
http

https://referencedata.trayport.com/instruments
Duration: 1.826s
https://analytics.trayport.com/api/orders/book?from=2025-06-04T06%3A00%3A00Z&until=2025-06-04T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=260&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 3.810s
https://referencedata.trayport.com/instruments
Duration: 1.568s
https://analytics.trayport.com/api/orders/book?from=2025-06-05T06%3A00%3A00Z&until=2025-06-05T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=260&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.311s
https://referencedata.trayport.com/instruments
Duration: 1.359s
https://analytics.trayport.com/api/orders/book?from=2025-06-06T06%3A00%3A00Z&until=2025-06-06T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=260&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.367s
http

https://referencedata.trayport.com/instruments
Duration: 2.781s
https://analytics.trayport.com/api/orders/book?from=2025-07-09T06%3A00%3A00Z&until=2025-07-09T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=261&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.373s
https://referencedata.trayport.com/instruments
Duration: 3.126s
https://analytics.trayport.com/api/orders/book?from=2025-07-10T06%3A00%3A00Z&until=2025-07-10T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=261&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.161s
https://referencedata.trayport.com/instruments
Duration: 1.538s
https://analytics.trayport.com/api/orders/book?from=2025-07-11T06%3A00%3A00Z&until=2025-07-11T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=261&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.732s
http

In [5]:
params_dict = {}

params_dict['tenor_list'] = ['q']
params_dict['tn1_list'] = [1]
params_dict['mkt_list'] = ['de'] * len(params_dict['tenor_list'])
params_dict['tn2_list'] = []
params_dict['prod'] = 'base'
params_dict['venue_list'] = ['eex']*len(params_dict['mkt_list'])
params_dict['start_date'] = datetime(2025, 2, 26)
params_dict['end_date'] = datetime(2025, 7, 16)
params_dict['ns'] = 2

# Fetch trades and best orders for the curve
assembler = TPDataAssembly(source='trayport', user='matej')
# assembler.set_start_end_time(start=[10,0,0], end=[12,0,0])    
trades_dict = assembler.get_data(params_dict, target_data='trades')
#assembler.set_data_source('database')
ba_dict = assembler.get_data(params_dict, target_data='best_orders')

assert ba_dict.keys() == trades_dict.keys(), "Curve doesn't fit for both trades and best_orders"

trades = pd.DataFrame()
ba = pd.DataFrame()
products = []
for key in trades_dict.keys():
    ba_aux = ba_dict[key].copy()
    trade_aux = trades_dict[key].copy()
    trade_aux.columns = [a + '_' + key for a in trade_aux.columns]
    ba_aux.columns = [a + '_' + key for a in ba_aux.columns]
    if trades.empty:
        trades = trade_aux.copy()
    else:
        trades = pd.concat([trades, trade_aux])
    if ba.empty:
        ba = ba_aux.copy()
    else:
        ba = pd.concat([ba, ba_aux])
    products.append(key)
trades.sort_index(inplace=True)
ba.sort_index(inplace=True)
ba.index.name='datetime'
trades.index.name ='datetime'


ti_inst = TI(trades, ba, products)

ti_inst.prepare_data()


data_raw = ti_inst.data

data = data_raw[~(data_raw['price_deq1'].notnull() & (data_raw['broker_id_deq1'] != 1441))][['price_deq1', 'volume_deq1','bidbestprice_deq1',
                  'askbestprice_deq1', 'mid_deq1', 'trade_side_deq1']].copy()

# data = data_raw[['price_deq1', 'volume_deq1','bidbestprice_deq1',
#                    'askbestprice_deq1', 'mid_deq1', 'trade_side_deq1']].copy()
    
data.columns = [a.split('_')[0] for a in data.columns]
data.columns = ['trd_price', 'volume', 'bid_price', 'ask_price', 'mid_price', 'trd_side']

data.to_csv(r's:\Algo\Files\andrej\Data\int_data_lag_production_sample_deq1.csv')

https://referencedata.trayport.com/instruments
Duration: 1.340s
https://analytics.trayport.com/api/trades?from=2025-02-26T07%3A00%3A00Z&until=2025-03-27T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000105&sequenceItemId=86&ContractType=SinglePeriod
Duration: 1.506s
https://referencedata.trayport.com/instruments
Duration: 1.254s
https://analytics.trayport.com/api/trades?from=2025-03-28T07%3A00%3A00Z&until=2025-04-29T07%3A00%3A00Z&instrumentId=10641710&sequenceId=10000105&sequenceItemId=87&ContractType=SinglePeriod
Duration: 2.280s
https://referencedata.trayport.com/instruments
Duration: 1.268s
https://analytics.trayport.com/api/trades?from=2025-04-29T07%3A00%3A00Z&until=2025-05-31T07%3A00%3A00Z&instrumentId=10641710&sequenceId=10000105&sequenceItemId=87&ContractType=SinglePeriod
Duration: 1.342s
https://referencedata.trayport.com/instruments
Duration: 1.241s
https://analytics.trayport.com/api/trades?from=2025-05-31T07%3A00%3A00Z&until=2025-06-26T18%3A00%3A00Z&instrumentId=10641710&s

https://referencedata.trayport.com/instruments
Duration: 1.487s
https://analytics.trayport.com/api/orders/book?from=2025-03-27T07%3A00%3A00Z&until=2025-03-27T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000105&sequenceItemId=86&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 5.667s
https://referencedata.trayport.com/instruments
Duration: 1.322s
https://analytics.trayport.com/api/orders/book?from=2025-03-28T07%3A00%3A00Z&until=2025-03-28T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000105&sequenceItemId=87&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.584s
https://referencedata.trayport.com/instruments
Duration: 1.366s
https://analytics.trayport.com/api/orders/book?from=2025-03-31T06%3A00%3A00Z&until=2025-03-31T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000105&sequenceItemId=87&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.352s
https:/

https://referencedata.trayport.com/instruments
Duration: 1.315s
https://analytics.trayport.com/api/orders/book?from=2025-05-01T06%3A00%3A00Z&until=2025-05-01T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000105&sequenceItemId=87&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 1.457s
https://referencedata.trayport.com/instruments
Duration: 1.184s
https://analytics.trayport.com/api/orders/book?from=2025-05-02T06%3A00%3A00Z&until=2025-05-02T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000105&sequenceItemId=87&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 6.516s
https://referencedata.trayport.com/instruments
Duration: 1.553s
https://analytics.trayport.com/api/orders/book?from=2025-05-05T06%3A00%3A00Z&until=2025-05-05T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000105&sequenceItemId=87&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.289s
https:/

https://referencedata.trayport.com/instruments
Duration: 1.594s
https://analytics.trayport.com/api/orders/book?from=2025-06-05T06%3A00%3A00Z&until=2025-06-05T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000105&sequenceItemId=87&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 6.158s
https://referencedata.trayport.com/instruments
Duration: 1.069s
https://analytics.trayport.com/api/orders/book?from=2025-06-06T06%3A00%3A00Z&until=2025-06-06T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000105&sequenceItemId=87&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.201s
https://referencedata.trayport.com/instruments
Duration: 1.310s
https://analytics.trayport.com/api/orders/book?from=2025-06-09T06%3A00%3A00Z&until=2025-06-09T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000105&sequenceItemId=87&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 5.904s
https:/

https://referencedata.trayport.com/instruments
Duration: 1.534s
https://analytics.trayport.com/api/orders/book?from=2025-07-10T06%3A00%3A00Z&until=2025-07-10T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000105&sequenceItemId=88&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 3.936s
https://referencedata.trayport.com/instruments
Duration: 1.204s
https://analytics.trayport.com/api/orders/book?from=2025-07-11T06%3A00%3A00Z&until=2025-07-11T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000105&sequenceItemId=88&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.430s
https://referencedata.trayport.com/instruments
Duration: 1.463s
https://analytics.trayport.com/api/orders/book?from=2025-07-14T06%3A00%3A00Z&until=2025-07-14T18%3A00%3A00Z&instrumentId=10641710&sequenceId=10000105&sequenceItemId=88&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.603s
https:/

In [13]:
data[(data.index.month==1)|(data.index.month==2)].to_csv(r's:\Algo\Files\andrej\Data\int_data_lag_dey1_jan_feb.csv')

In [14]:
data[(data.index.month==3)|(data.index.month==4)].to_csv(r's:\Algo\Files\andrej\Data\int_data_lag_dey1_mar_apr.csv')

In [15]:
data[(data.index.month==5)|(data.index.month==6)].to_csv(r's:\Algo\Files\andrej\Data\int_data_lag_dey1_may_jun.csv')

In [16]:
data[(data.index.month==7)|(data.index.month==8)].to_csv(r's:\Algo\Files\andrej\Data\int_data_lag_dey1_july_aug.csv')

## Using Our database

In [12]:
data_class = TPData()
data_class.create_connection('OracleSQL')
data_class_pg = TPData()
data_class_pg.create_connection('PostgreSQL')
data_class_tp = TPDataDa()

mkt_list = ['de']
tenor_list = ['w']
tn_list = [1]
prod = 'base'
venue_list = ['eex']
# start_date = datetime(2024, 6, 3)
start_date = datetime(2024, 7, 1)
end_date = datetime(2025, 6, 30)

allwd_broker_ids = [1441]

sample_dates = pd.date_range(start_date, end_date, freq='B')

n = 16
n_t = 15
d_t = 1
date_range_dict = {k.date(): pd.date_range(k, periods=n, freq='B')
                   for k in sample_dates[:-n+1]}

n_s = 2

dates = pd.date_range(start_date, end_date, freq='B')
product_date = [dates.shift(1, freq='B') if t == 'da' else
                dates.shift(1, freq='D') if t == 'd' else
                dates.shift(tn, freq='W-MON') if t == 'w' else
                (dates + n_s * dates.freq).shift(tn, freq='2QS-Apr') if t in ['sum', 'win'] else
                (dates + n_s * dates.freq).shift(tn, freq='YS') if t in ['dec'] else
                (dates + n_s * dates.freq).shift(tn, freq=t.upper() + 'S')
                for t, tn in zip(tenor_list, tn_list)]

start_time = time(11, 0, 0)
end_time = time(14, 0, 0)

tr_data_dict = {m + t + str(n): [] for m, t, n in zip(mkt_list, tenor_list, tn_list)}
ba_data_dict = {m + t + str(n): [] for m, t, n in zip(mkt_list, tenor_list, tn_list)}
agg_dict = {'price': 'sum', 'volume': 'sum', 'action': 'median',
            'broker_id': 'median', 'count': 'sum'}

for m, t, n, p_dates in zip(mkt_list, tenor_list, tn_list, product_date):
    df_tr, df_ba = pd.DataFrame([]), pd.DataFrame([])
    series = pd.Series(p_dates, index=dates)
    for p_d, ds in series.groupby(series).groups.items():
        bT = datetime.combine(ds[0], start_time)
        eT = datetime.combine(ds[-1], end_time)
        # Trades
        df_tr_aux = data_class.get_trades(m, t, venue_list, p_d, bT, eT, prod)
        # Filter by broker
        if not allwd_broker_ids or t == 'da':
            pass
        else:
            df_tr_aux = df_tr_aux[df_tr_aux['broker_id'].isin(allwd_broker_ids)]
        # Clean trades
        df_ba_aux = data_class_pg.get_best_ob_data(m, t, venue_list, p_d, bT, eT, prod, None, aonn=False)
        if df_ba_aux.empty:
            df_ba_aux = data_class_tp.get_best_ob_data(m, t, venue_list, p_d, bT, eT, prod, aonn=False)
        df_ba_aux = df_ba_aux.rename(columns={'bidbestprice': 'b_price', 'askbestprice': 'a_price'})
        df_tr_aux = data_class.clean_trades(df_tr_aux, df_ba_aux)
        try:
            df_tr_aux = df_tr_aux.between_time(start_time, end_time)
        except(TypeError):
            pass
        # Group trades
        df_tr_aux['count'] = 1
        #df_tr_aux['price'] *= df_tr_aux['volume']
        #df_tr_aux = df_tr_aux.groupby(df_tr_aux.index).agg(agg_dict)
        #df_tr_aux['price'] /= df_tr_aux['volume']
        df_tr = pd.concat([df_tr, df_tr_aux])
        df_ba = pd.concat([df_ba, df_ba_aux])
        del df_tr_aux, df_ba_aux
    tr_data_dict[m + t + str(n)] = df_tr
    ba_data_dict[m + t + str(n)] = df_ba
    
trades2=pd.concat({k: v for k, v in tr_data_dict.items()}, axis=1)
trades2.columns = ['_'.join([col[-1], col[0]]) for col in trades2.columns]
trades2.index.name='datetime'
trades2=trades2[['price_dew1', 'volume_dew1', 'action_dew1', 'broker_id_dew1']]

ba2=pd.concat({k: v for k, v in ba_data_dict.items()}, axis=1)
ba2.columns = ['bidbestprice_'+ba2.columns[0][0], 'askbestprice_'+ba2.columns[0][0]]

trades2.sort_index(inplace=True)
ba2.sort_index(inplace=True)
ba2.index.name='datetime'
trades2.index.name ='datetime'

products2=[''.join(map(str,(mkt_list+tenor_list+tn_list)))]

ti_inst = TI(trades2, ba2, products2)

ti_inst.prepare_data()


data_raw = ti_inst.data

data = data_raw[~(data_raw['price_dew1'].notnull() & (data_raw['broker_id_dew1'] != 1441))][['price_dew1', 'volume_dew1','bidbestprice_dew1',
                  'askbestprice_dew1', 'mid_dew1', 'trade_side_dew1']].copy()

# data = data_raw[['price_dew1', 'volume_dew1','bidbestprice_dew1',
#                    'askbestprice_dew1', 'mid_dew1', 'trade_side_dew1']].copy()
    
data.columns = [a.split('_')[0] for a in data.columns]
data.columns = ['trd_price', 'volume', 'bid_price', 'ask_price', 'mid_price', 'trd_side']

data.to_csv(r's:\Algo\Files\andrej\Data\int_data_lead_production_sample_dew1_our_db.csv')

Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the database postgre
Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the database postgre
Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the database postgre
Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the database postgre
Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the database postgre
Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the database postgre
Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the database 

In [18]:
data_class = TPData()
data_class.create_connection('OracleSQL')
data_class_pg = TPData()
data_class_pg.create_connection('PostgreSQL')
data_class_tp = TPDataDa()

mkt_list = ['de']
tenor_list = ['m']
tn_list = [2]
prod = 'base'
venue_list = ['eex']
# start_date = datetime(2024, 6, 3)
start_date = datetime(2025, 2, 28)
end_date = datetime(2025, 3, 14)

allwd_broker_ids = [1441]

sample_dates = pd.date_range(start_date, end_date, freq='B')

n = 16
n_t = 15
d_t = 1
date_range_dict = {k.date(): pd.date_range(k, periods=n, freq='B')
                   for k in sample_dates[:-n+1]}

n_s = 2

dates = pd.date_range(start_date, end_date, freq='B')
product_date = [dates.shift(1, freq='B') if t == 'da' else
                dates.shift(1, freq='D') if t == 'd' else
                dates.shift(tn, freq='W-MON') if t == 'w' else
                (dates + n_s * dates.freq).shift(tn, freq='2QS-Apr') if t in ['sum', 'win'] else
                (dates + n_s * dates.freq).shift(tn, freq='YS') if t in ['dec'] else
                (dates + n_s * dates.freq).shift(tn, freq=t.upper() + 'S')
                for t, tn in zip(tenor_list, tn_list)]

start_time = time(9, 0, 0)
end_time = time(18, 0, 0)

tr_data_dict = {m + t + str(n): [] for m, t, n in zip(mkt_list, tenor_list, tn_list)}
ba_data_dict = {m + t + str(n): [] for m, t, n in zip(mkt_list, tenor_list, tn_list)}
agg_dict = {'price': 'sum', 'volume': 'sum', 'action': 'median',
            'broker_id': 'median', 'count': 'sum'}

for m, t, n, p_dates in zip(mkt_list, tenor_list, tn_list, product_date):
    df_tr, df_ba = pd.DataFrame([]), pd.DataFrame([])
    series = pd.Series(p_dates, index=dates)
    for p_d, ds in series.groupby(series).groups.items():
        bT = datetime.combine(ds[0], start_time)
        eT = datetime.combine(ds[-1], end_time)
        # Trades
        df_tr_aux = data_class.get_trades(m, t, venue_list, p_d, bT, eT, prod)
        # Filter by broker
        if not allwd_broker_ids or t == 'da':
            pass
        else:
            df_tr_aux = df_tr_aux[df_tr_aux['broker_id'].isin(allwd_broker_ids)]
        # Clean trades
        df_ba_aux = data_class_pg.get_best_ob_data(m, t, venue_list, p_d, bT, eT, prod, None, aonn=False)
        if df_ba_aux.empty:
            df_ba_aux = data_class_tp.get_best_ob_data(m, t, venue_list, p_d, bT, eT, prod, None, aonn=False)
        df_ba_aux = df_ba_aux.rename(columns={'bidbestprice': 'b_price', 'askbestprice': 'a_price'})
        df_tr_aux = data_class.clean_trades(df_tr_aux, df_ba_aux)
        try:
            df_tr_aux = df_tr_aux.between_time(start_time, end_time)
        except(TypeError):
            pass
        # Group trades
        df_tr_aux['count'] = 1
        #df_tr_aux['price'] *= df_tr_aux['volume']
        #df_tr_aux = df_tr_aux.groupby(df_tr_aux.index).agg(agg_dict)
        #df_tr_aux['price'] /= df_tr_aux['volume']
        df_tr = pd.concat([df_tr, df_tr_aux])
        df_ba = pd.concat([df_ba, df_ba_aux])
        del df_tr_aux, df_ba_aux
    tr_data_dict[m + t + str(n)] = df_tr
    ba_data_dict[m + t + str(n)] = df_ba
    
trades2=pd.concat({k: v for k, v in tr_data_dict.items()}, axis=1)
trades2.columns = ['_'.join([col[-1], col[0]]) for col in trades2.columns]
trades2.index.name='datetime'
trades2=trades2[['price_dem2', 'volume_dem2', 'action_dem2', 'broker_id_dem2']]

ba2=pd.concat({k: v for k, v in ba_data_dict.items()}, axis=1)
ba2.columns = ['bidbestprice_'+ba2.columns[0][0], 'askbestprice_'+ba2.columns[0][0]]

trades2.sort_index(inplace=True)
ba2.sort_index(inplace=True)
ba2.index.name='datetime'
trades2.index.name ='datetime'

products2=[''.join(map(str,(mkt_list+tenor_list+tn_list)))]

ti_inst = TI(trades2, ba2, products2)

ti_inst.prepare_data()


data_raw = ti_inst.data

data = data_raw[~(data_raw['price_dem2'].notnull() & (data_raw['broker_id_dem2'] != 1441))][['price_dem2', 'volume_dem2','bidbestprice_dem2',
                  'askbestprice_dem2', 'mid_dem2', 'trade_side_dem2']].copy()

# data = data_raw[['price_dem2', 'volume_dem2','bidbestprice_dem2',
#                    'askbestprice_dem2', 'mid_dem2', 'trade_side_dem2']].copy()
    
data.columns = [a.split('_')[0] for a in data.columns]
data.columns = ['trd_price', 'volume', 'bid_price', 'ask_price', 'mid_price', 'trd_side']

data.to_csv(r's:\Algo\Files\andrej\Data\int_data_lag_production_sample_dem2_our_db.csv')

Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the database postgre


# Parsing calibration results new

### With Martinovo Zatvaranie

In [19]:
# Load the pickled object
import pickle
with open(r'c:\Users\andrej\Projects\EnergyTrading\Python\Strategies\BA_convergence\out_dict_dey1.pkl', 'rb') as file:
    loaded_object = pickle.load(file)

df=pd.DataFrame([(k[0],k[1],k[2], k[3], k[4],k[5],k[6],k[10], v[0],v[1],v[2], v[3], v[4], v[5]) for k,v in loaded_object.items()], columns=[
                                                                                          'BA_large',
                                                                                          'BA_small',
                                                                                          'BA_vol',
                                                                                          'BA_thres',
                                                                                          'MACD_l',
                                                                                          'MACD_s',
                                                                                          'Intensity',

                                                                                            
                                                                                          'Trail_Stop',

                                                                                          
                                                                                           'cumpnl', 'sharp', 'max_drawdown', 'num_trades', 'trade_per_day', 'acc']).sort_values(by=['cumpnl'], ascending=False)

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\andrej\\Projects\\EnergyTrading\\Python\\Strategies\\BA_convergence\\out_dict_dey1.pkl'

In [ ]:
filter1=True#df['Trail_Stop']==100
filter2=True#df['TP']>df['SL'].apply(abs)
filter3=True
filter4=True#df['MACD_l']==-1*df['MACD_s']
filter5=df['num_trades']>40


filter=filter1&filter2&filter3&filter4&filter5


df[filter][0:50]

In [ ]:
b.ll_aux_ens_deq1_fair_price,
b.ll_aux_ens_deq1_fair_ret,
b.obook_a_price_deq1,
b.obook_a_price_sparsity_deq1,
b.obook_a_vol_deq1,
b.obook_b_price_deq1,
b.obook_b_price_sparsity_deq1,
b.obook_b_vol_deq1,
b.obook_ba_spread_deq1,
b.obook_ba_volrat_00_deq1,
b.obook_ba_volrat_30_deq1,
b.obook_ba_volrat_50_deq1,
b.obook_ba_volrat_90_deq1,
b.obook_mid_priceW_00_deq1,
b.obook_mid_priceW_30_deq1,
b.obook_mid_priceW_50_deq1,
b.obook_mid_priceW_90_deq1,
b.obook_mid_priceW_d_00_deq1,
b.obook_mid_priceW_d_30_deq1,
b.obook_mid_priceW_d_50_deq1,
b.obook_mid_priceW_d_90_deq1,
b.obook_mid_price_deq1,
b.vpin_100_deq1,
b.vpin_10_deq1,
b.vpin_20_deq1,
b.vpin_50_deq1